[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-4-llms-genai/09-fine-tuning-decision-and-data/code/notebook.ipynb)

# Class 4.9: Fine-tuning I, the decision and the data

Fine-tuning changes a model's **behavior**, not its facts. Here we prepare a small
supervised fine-tuning (SFT) dataset that teaches the Domain Knowledge Assistant a
consistent **house style**: concise answers that always cite the IRS publication
and tax year, and that decline politely when the answer is not covered. We inspect
it, format it with the model's chat template, split it, and define the LoRA and
4-bit (QLoRA) training configuration.

**What we will cover:** load and inspect the dataset, the chat format, a train/validation
split kept disjoint from the class 4.8 eval set, and the LoRA + QLoRA + trainer config.
We stop before training; class 4.10 runs the QLoRA job.

## Setup

New libraries for this class (fine-tuning). `bitsandbytes` needs a GPU, so run this
on a free-tier Colab **T4 GPU (16 GB)** runtime.

Install the necessary libraries in your virtual environment:

`pip install transformers datasets peft trl bitsandbytes accelerate`

## 1. The decision: behavior, not facts

We are not teaching the model new tax numbers (that is RAG's job, class 4.5). We
are teaching it a consistent way to answer: short, plain, always cite the pub and
year, and say "I do not find that in the documents" when it should. That is a
behavior, exactly what fine-tuning is good at.

We call that target style the model's **house style**. The term is borrowed from
publishing, where a "house style" is the fixed set of rules an outlet applies to
everything it prints (tone, formatting, how sources are cited). Here it is our
assistant's rules: concise, always cite the publication and year, and decline
politely when the answer is not covered.

## 2. Load and inspect the dataset

In [1]:
import json
from datasets import Dataset

rows = [json.loads(l) for l in open("data/dataset.jsonl")]
ds = Dataset.from_list(rows)
print(len(ds), "examples")

# Each row is a chat conversation: system rule, user question, assistant target.
ex = ds[0]["messages"]
for m in ex:
    print(f"[{m['role']}] {m['content']}")

c:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-ai-july-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


72 examples
[system] You are the Domain Knowledge Assistant. Answer in one or two sentences, in plain language, and cite the IRS publication and tax year in parentheses, like (Pub 501, 2025). If the answer is not in the documents, reply exactly: "I do not find that in the documents."
[user] For 2025, what standard deduction does a single filer take?
[assistant] A single filer's 2025 basic standard deduction is 15,000 (Pub 501, 2025).


## 3. Format with the chat template

The trainer turns each conversation into the exact text the model sees at
inference, using the model's own chat template. We only load the tokenizer here to
show that formatting; the 4-bit model itself is loaded in class 4.10.

In [2]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # small, open, fits the free Colab T4 (16 GB) GPU
tok = AutoTokenizer.from_pretrained(MODEL_ID)

# apply_chat_template renders the messages into one training string.
formatted = tok.apply_chat_template(ds[0]["messages"], tokenize=False)
print(formatted)

c:\Users\Sourav Karmakar\Desktop\Work\LogicMojo\logicmojo-ai-july-2026\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sourav Karmakar\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


<|im_start|>system
You are the Domain Knowledge Assistant. Answer in one or two sentences, in plain language, and cite the IRS publication and tax year in parentheses, like (Pub 501, 2025). If the answer is not in the documents, reply exactly: "I do not find that in the documents."<|im_end|>
<|im_start|>user
For 2025, what standard deduction does a single filer take?<|im_end|>
<|im_start|>assistant
A single filer's 2025 basic standard deduction is 15,000 (Pub 501, 2025).<|im_end|>



## 4. Train and validation split

Split the examples into a training set (the model learns from) and a small
validation set (we watch it during training). The class 4.8 eval set is a separate,
held-out test and is never trained on.

In [3]:
split = ds.train_test_split(test_size=0.2, seed=42)  # hold out 20% as validation; seed=42 fixes the shuffle so the split is reproducible
train_ds, val_ds = split["train"], split["test"]
print("train:", len(train_ds), "| val:", len(val_ds))

train: 57 | val: 15


## 5. LoRA configuration

LoRA freezes the base weights and trains a small low-rank add-on (`W' = W + BA`).
`r` is the rank of that add-on; `lora_alpha` scales it; `target_modules` picks
which layers get an adapter (the attention projections here).

LoRA is **not** applied to every weight in the model, you choose which matrices
get an adapter via `target_modules`. The usual choice is the four attention
projections `q_proj, k_proj, v_proj, o_proj` (our default, a strong cheap
baseline). Adding the feed-forward/MLP projections (`gate, up, down`) reaches
more of the network for harder behavior shifts, at the cost of a larger adapter
and more overfit risk; embeddings, the output head, and layer-norms are normally
left alone. Two dials set the adapter size: the rank `r` and how many modules you
target. Even a 7-8B base at `r=16` trains only ~10-14M parameters (well under 1%).

In [4]:
from peft import LoraConfig

lora = LoraConfig(
    r=16,                # rank of the low-rank add-on B,A (the main capacity/size knob; higher = more it can learn, larger adapter, more overfit risk)
    lora_alpha=32,       # scaling: the update is applied as (alpha/r)*BA, so 32/16 = 2x here; convention is alpha = 2*r
    lora_dropout=0.05,   # drop 5% of adapter activations while training; light regularization on a small dataset
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # which weight matrices get an adapter: the attention query/key/value/output projections (add the MLP layers for more capacity)
    task_type="CAUSAL_LM",  # next-token language model, so PEFT wires the adapter in correctly
)
print(lora)

W0921 21:56:34.912000 16904 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'v_proj', 'o_proj', 'q_proj', 'k_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


## 6. 4-bit quantization (the QLoRA part)

Store the frozen base model in 4-bit so it barely uses memory, then train the LoRA
add-on on top. `nf4` is the 4-bit number format; double quantization saves a little
more; the compute dtype is what the math runs in.

In [5]:
import torch
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,                      # store the frozen base weights in 4-bit (~4x smaller than 16-bit)
    bnb_4bit_quant_type="nf4",              # NormalFloat4: a 4-bit format tuned to the bell-curve spread of weights; more accurate than plain int4
    bnb_4bit_use_double_quant=True,         # also quantize the quantization constants; a little more memory saved at no real quality cost
    bnb_4bit_compute_dtype=torch.bfloat16,  # weights are stored in 4-bit but dequantized to bfloat16 for the actual matmuls, so the math stays stable
)
print(bnb)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



## 7. Training configuration (defined, not run)

Set the knobs for the run: how many epochs, the batch size and gradient
accumulation (to fit a small GPU), and the learning rate. We define it here and
stop. Class 4.10 passes the model, `lora`, `bnb`, this config, and the split to an
`SFTTrainer` and calls `train()`.

In [6]:
from trl import SFTConfig

cfg = SFTConfig(
    output_dir="dka-lora",             # where the trained adapter is written; "dka" = Domain Knowledge Assistant (the Module 4 milestone), "lora" = a LoRA adapter
    num_train_epochs=3,                # passes over the training data; on ~57 examples, 3 is a sensible start before overfitting
    per_device_train_batch_size=2,     # examples per GPU step; kept small for limited VRAM
    gradient_accumulation_steps=4,     # accumulate 4 steps before updating, so the effective batch is 2*4 = 8 without the memory for 8 at once
    learning_rate=2e-4,                # LoRA tolerates a higher LR than full fine-tuning (only the small adapter trains); 1e-4 to 3e-4 is typical
    logging_steps=5,                   # print the training loss every 5 steps
    save_strategy="epoch",             # save a checkpoint at the end of each epoch
)
print(cfg.num_train_epochs, "epochs |", "lr", cfg.learning_rate)

# NOTE: we do NOT train here. In class 4.10:
#   from trl import SFTTrainer
#   from transformers import AutoModelForCausalLM
#   model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
#   trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds,
#                        eval_dataset=val_ds, peft_config=lora)
#   trainer.train()

3 epochs | lr 0.0002


## Recap

You prepared a house-style dataset (behavior, not facts), formatted it with the
chat template, split it disjoint from the 4.8 eval set, and defined the LoRA, 4-bit,
and trainer configuration. In class 4.10 (Fine-tuning II) we run the QLoRA job on a
free Colab T4 GPU, then evaluate the tuned model against the base on the 4.8 eval set,
merge, and serve.